# 01 — GEE Data Extraction
## Nyando Flood AI · James Koero · Kisumu, Kenya

**Goal:** Extract real satellite features for the Nyando River Basin using Google Earth Engine.

**Why this notebook matters:** All 6 input features come from open, reproducible satellite datasets.
Anyone can re-run this notebook to refresh the data or adapt it to another basin.

**Data sources:**
- Elevation & slope: NASA NASADEM 30m
- Rainfall: CHIRPS v2 daily (3-day sum around April 2024 flood)
- Flood labels: Sentinel-1 SAR (VV backscatter < -16 dB threshold)
- Soil clay: ISRIC SoilGrids 250m (0-5cm)
- River distance: HydroSHEDS + OSM
- Land cover: ESA WorldCover 2021 10m

**Study area:** lon 34.70–35.40°E, lat 0.40°S–0.10°N (Nyando sub-county, Kisumu County, Kenya)

**Output:** `data/training/nyando_training_v1_raw_gee.csv` — 2,308 observation points

In [ ]:
# Install GEE dependencies (Colab)
!pip install earthengine-api geemap -q
print('Dependencies installed ✅')

### Authenticate GEE
You need a Google account with GEE access at earthengine.google.com

In [ ]:
import ee
ee.Authenticate()
ee.Initialize(project='ee-jmskoero')  # replace with your GEE project
print('GEE authenticated ✅')

### Define Study Area
Nyando River Basin boundaries — chosen to cover 42 wards of Nyando sub-county.

In [ ]:
import numpy as np, pandas as pd

# Nyando Basin bounding box
nyando = ee.Geometry.Rectangle([34.70, -0.40, 35.40, 0.10])

# Random sample grid — 2308 points, reproducible seed
sample_pts = ee.FeatureCollection.randomPoints(region=nyando, points=2308, seed=42)
print('Sample grid: 2,308 points over Nyando Basin ✅')

### Build Feature Stack
Each band in this stack is one input feature for the flood model.

In [ ]:
# Elevation + slope (NASA NASADEM)
dem   = ee.Image('NASA/NASADEM_HGT/001').select('elevation')
slope = ee.Terrain.slope(dem)

# Rainfall — 3-day sum (CHIRPS, April 2024 flood event)
rainfall = ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY') \
    .filterDate('2024-04-20', '2024-04-24').sum().rename('rainfall_3day')

# Soil clay 0-5cm (ISRIC SoilGrids)
clay = ee.Image('projects/soilgrids-isric/clay_mean').select('b0').rename('clay_percent')

# Land cover (ESA WorldCover 2021)
lc = ee.ImageCollection('ESA/WorldCover/v200').first().rename('land_cover')

# River distance (HydroSHEDS flow accumulation proxy)
hydro    = ee.Image('WWF/HydroSHEDS/15ACC').rename('acc')
dist_riv = hydro.gte(500).Not().fastDistanceTransform().rename('distance_river')

# Population (WorldPop 2020)
pop = ee.ImageCollection('WorldPop/GP/100m/pop') \
    .filter(ee.Filter.eq('country','KEN')) \
    .filter(ee.Filter.eq('year',2020)).first().rename('population')

# Sentinel-1 SAR flood label (April 2024)
s1 = ee.ImageCollection('COPERNICUS/S1_GRD') \
    .filterDate('2024-04-18','2024-04-28') \
    .filterBounds(nyando) \
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation','VV')) \
    .filter(ee.Filter.eq('instrumentMode','IW')) \
    .select('VV').mean()
flood = s1.lt(-16.0).rename('flooded').toInt()

# Stack all bands
stack = dem.addBands(slope).addBands(rainfall).addBands(dist_riv) \
            .addBands(clay).addBands(lc).addBands(pop).addBands(flood)
print('Feature stack ready ✅')

### Sample and Export
This takes ~2 minutes. The `getInfo()` call pulls data from GEE servers.

In [ ]:
print('Sampling (2-3 min)...')
sampled = stack.sampleRegions(collection=sample_pts, scale=100, geometries=True, tileScale=4)
info = sampled.getInfo()

records = []
for feat in info['features']:
    p = feat['properties']; c = feat['geometry']['coordinates']
    records.append({'lon':round(c[0],6),'lat':round(c[1],6),
        'elevation':p.get('elevation',np.nan),'slope':round(p.get('slope',np.nan),2),
        'rainfall_3day':round(p.get('rainfall_3day',np.nan),1),
        'distance_river':round(p.get('distance_river',np.nan),0),
        'clay_percent':round(p.get('clay_percent',np.nan),1),
        'land_cover':p.get('land_cover',np.nan),'population':p.get('population',0),
        'flooded':int(p.get('flooded',0))})

df = pd.DataFrame(records).dropna(subset=['elevation','slope','rainfall_3day'])
df.to_csv('/content/nyando_training_v1_raw_gee.csv', index=False)

print(f'Saved: {len(df)} points')
print(f'Flood rate: {df["flooded"].mean():.1%}')
print(f'Elevation: {df.elevation.min():.0f}–{df.elevation.max():.0f}m')
print(f'Rainfall: {df.rainfall_3day.min():.1f}–{df.rainfall_3day.max():.1f}mm')
print('\nDownload nyando_training_v1_raw_gee.csv from Colab file browser ✅')